# 01 Stratified Split — per-bucket train/val (no embedding required)

Cheap replacement for the cluster-based split in `data_preparation/00_part2b_embed_cluster_split.ipynb`.
Uses fields that already exist in each bucket parquet — **no GPU, no embedding**.

**Stratification axis:** `subcat_aggregated` × `SALES_RANK` quartile (computed **within each bucket**, so per-gender automatically since each bucket is single-gender).

**Split:** stratified 50/50 at the ASIN level, seed=42.

**No leakage** — stratification keys are product attributes; no model is fit.

**Inputs:**  `data/{bucket}/amazon_shoes_8_combined.parquet` (the 6 outputs of `00_brand_filter.ipynb`)

**Outputs:** `code/{bucket}/asin_split.csv` with columns `[ASIN, stratum, split]`

Downstream: after running `data_preparation/00_part1_data_preparation.ipynb` for a bucket, materialize train/val parquets with the snippet at the bottom of this notebook (or fold it into part1).

In [1]:
import pandas as pd
import numpy as np
import random
from pathlib import Path

def _find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data_preparation').exists() and (p / 'code').exists():
            return p
    raise RuntimeError(
        'Cannot find project root.\n'
        'Expected a directory containing both "data_preparation/" and "code/".\n'
        f'Searched from: {Path.cwd()}'
    )

PROJECT_ROOT = _find_project_root()
DATA_ROOT    = PROJECT_ROOT / 'data'

print(f'PROJECT_ROOT : {PROJECT_ROOT}')

PROJECT_ROOT : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate


## Config

`BUCKETS` is the 6 directories produced by `00_brand_filter.ipynb`. `MIN_STRATUM_SIZE` is the threshold below which a stratum is too small for a clean 50/50 split and gets merged into `OTHER_SMALL`.

In [2]:
BUCKETS = [
    # New buckets needing a split: adidas-women (37, >36 so its own) + the pooled
    # <=36 well-known leftovers per gender. The >36 individual brand buckets
    # (women_asics/reebok/heydude/dr_scholl, men_adidas/asics/reebok/heydude/olukai)
    # already have asin_split.csv, so they are left out here.
    'women_adidas',
    'women_rest',
    'men_rest',
]

SEED              = 42
N_QUARTILES       = 4
MIN_STRATUM_SIZE  = 4
TRAIN_FRAC        = 0.8

print(f'Buckets          : {len(BUCKETS)}')
print(f'Seed             : {SEED}')
print(f'SALES_RANK bins  : {N_QUARTILES}')
print(f'Min stratum size : {MIN_STRATUM_SIZE}  (smaller strata merged → OTHER_SMALL)')
print(f'Train fraction   : {TRAIN_FRAC}')

Buckets          : 11
Seed             : 42
SALES_RANK bins  : 4
Min stratum size : 4  (smaller strata merged → OTHER_SMALL)
Train fraction   : 0.8


## Stratify and split each bucket

In [3]:
def stratified_split(bucket: str, seed: int = SEED) -> pd.DataFrame:
    """ASIN-level stratified split (TRAIN_FRAC, currently 80/20) for one bucket.

    Returns DataFrame [ASIN, stratum, split] and prints a per-stratum audit.
    """
    raw_path = DATA_ROOT / bucket / 'amazon_shoes_8_combined.parquet'
    assert raw_path.exists(), f'Missing {raw_path}'

    df = pd.read_parquet(raw_path)

    # ASIN-level features for stratification
    asin_feat = (
        df.groupby('ASIN')
          .agg(
              subcat_aggregated=('subcat_aggregated', 'first'),
              mean_sales_rank=('SALES_RANK', 'mean'),
          )
          .reset_index()
    )

    # Fill any ASIN with all-NaN SALES_RANK with bucket median
    bucket_median = asin_feat['mean_sales_rank'].median()
    asin_feat['mean_sales_rank'] = asin_feat['mean_sales_rank'].fillna(bucket_median)

    # SALES_RANK quartiles within this bucket
    try:
        asin_feat['rank_q'] = pd.qcut(
            asin_feat['mean_sales_rank'],
            q=N_QUARTILES,
            labels=[f'Q{i+1}' for i in range(N_QUARTILES)],
            duplicates='drop',
        ).astype(str)
    except ValueError:
        asin_feat['rank_q'] = pd.qcut(
            asin_feat['mean_sales_rank'],
            q=2,
            labels=['Q_lo', 'Q_hi'],
            duplicates='drop',
        ).astype(str)

    asin_feat['stratum'] = (
        asin_feat['subcat_aggregated'].astype(str) + ' | ' + asin_feat['rank_q']
    )

    # Merge tiny strata
    sizes = asin_feat['stratum'].value_counts()
    small = sizes[sizes < MIN_STRATUM_SIZE].index.tolist()
    if small:
        asin_feat.loc[asin_feat['stratum'].isin(small), 'stratum'] = 'OTHER_SMALL'

    rng = random.Random(seed)
    rows = []
    print(f'\n--- {bucket} ---')
    print(f'  {"stratum":<35} {"total":>6} {"train":>6} {"val":>6}')
    for stratum, group in asin_feat.groupby('stratum', sort=False):
        asins = sorted(group['ASIN'].tolist())
        rng.shuffle(asins)
        n_train = max(1, round(len(asins) * TRAIN_FRAC))
        for a in asins[:n_train]:
            rows.append({'ASIN': a, 'stratum': stratum, 'split': 'train'})
        for a in asins[n_train:]:
            rows.append({'ASIN': a, 'stratum': stratum, 'split': 'val'})
        print(f'  {stratum:<35} {len(asins):>6} {n_train:>6} {len(asins)-n_train:>6}')

    out = pd.DataFrame(rows)
    n_tr = (out['split'] == 'train').sum()
    n_vl = (out['split'] == 'val').sum()
    print(f'  {"TOTAL":<35} {n_tr+n_vl:>6} {n_tr:>6} {n_vl:>6}   ({n_tr/(n_tr+n_vl)*100:.1f}% train)')
    return out


summary_rows = []
for bucket in BUCKETS:
    split_df = stratified_split(bucket)

    out_dir  = DATA_ROOT / bucket / 'keys'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / 'asin_split.csv'
    split_df.to_csv(out_path, index=False)

    summary_rows.append({
        'bucket'  : bucket,
        'asins'   : len(split_df),
        'train'   : int((split_df['split'] == 'train').sum()),
        'val'     : int((split_df['split'] == 'val').sum()),
        'strata'  : split_df['stratum'].nunique(),
        'out_path': out_path.relative_to(PROJECT_ROOT).as_posix(),
    })

print()
print('=' * 78)
print('SUMMARY')
print('=' * 78)
print(pd.DataFrame(summary_rows).to_string(index=False))


--- women_nike ---
  stratum                              total  train    val
  Running | Q2                             7      6      1
  Running | Q3                            10      8      2
  OTHER_SMALL                              6      5      1
  Running | Q4                             4      3      1
  Shoes | Q4                              16     13      3
  Shoes | Q3                               8      6      2
  Shoes | Q2                              14     11      3
  Running | Q1                             8      6      2
  Shoes | Q1                              10      8      2
  TOTAL                                   83     66     17   (79.5% train)

--- women_asics ---
  stratum                              total  train    val
  Running | Q3                             9      7      2
  OTHER_SMALL                             10      8      2
  Running | Q4                             9      7      2
  Running | Q2                             5      4      1

## Downstream — materialize train/val parquets

After running `data_preparation/00_part1_data_preparation.ipynb` for a bucket, use the saved `asin_split.csv` to slice that bucket's cleaned parquet into train/val files. Add this as a final cell to `00_part1_*` (or run it as a tiny standalone notebook in place of the old `00_part2b_*`):

```python
import pandas as pd
from pathlib import Path

BRAND_BUCKET = 'women_top1_dream_pairs'   # change per run
DATA_DIR     = PROJECT_ROOT / 'data' / BRAND_BUCKET / 'amzn_shoes_monthly_diffs_ffill_fixed_splits'
KEY_DIR      = PROJECT_ROOT / 'data' / BRAND_BUCKET / 'keys'

clean    = pd.read_parquet(DATA_DIR / 'all_products_clean.parquet')
split    = pd.read_csv(KEY_DIR / 'asin_split.csv')
train_a  = set(split.loc[split['split'] == 'train', 'ASIN'])
val_a    = set(split.loc[split['split'] == 'val',   'ASIN'])

df_train = clean[clean['ASIN'].isin(train_a)].copy()
df_val   = clean[clean['ASIN'].isin(val_a)].copy()

df_train.to_parquet(DATA_DIR / 'train-00000-of-00001.parquet', index=False)
df_val.to_parquet(DATA_DIR / 'validation-00000-of-00001.parquet', index=False)

for name, dfx in [('train', df_train), ('val', df_val)]:
    keys = dfx[['ASIN', 'date']].copy()
    keys['date'] = pd.to_datetime(keys['date']).dt.strftime('%Y-%m-%d')
    keys.to_csv(KEY_DIR / f'main_{name}_keys.csv', index=False)

print(f'Train : {len(df_train):,} rows | {df_train["ASIN"].nunique()} ASINs')
print(f'Val   : {len(df_val):,} rows | {df_val["ASIN"].nunique()} ASINs')
```